# Implementação do solver usando o trabalho final de COS360 como base

## Imports

In [1]:
import pandas as pd
import pulp
import gcsfs
import os
from dotenv import load_dotenv, find_dotenv

## Paths

In [2]:
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

project_root = os.path.dirname(dotenv_path)

KEY_PATH_RELATIVE = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if KEY_PATH_RELATIVE:
    KEY_PATH_ABSOLUTE = os.path.join(project_root, KEY_PATH_RELATIVE)
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH_ABSOLUTE
    print(f"Credenciais carregadas com sucesso de: {KEY_PATH_ABSOLUTE}")
else:
    print(
        "Aviso: A variável GOOGLE_APPLICATION_CREDENTIALS não foi encontrada no arquivo .env"
    )
    print("O acesso ao GCS pode falhar.")

BUCKET_NAME = os.getenv('GCS_BUCKET_NAME')
INPUT_PATH = f"gs://{BUCKET_NAME}/solver_inputs/candidatos_solver_mg.csv"

Credenciais carregadas com sucesso de: g:\BackupC\Faculdade\UFRJ\TCC\analise-geografica-energia-solar-brasil\.secrets/tcc-usinas-solares-brasil-29b2750a20f8.json


## Carregamento de Dados

In [3]:
fs = gcsfs.GCSFileSystem()
with fs.open(INPUT_PATH) as f:
    df_candidatos = pd.read_csv(f, sep=';', decimal=',')

print(f"Base carregada com {len(df_candidatos):,} latifúndios.")
print(f"Potencial Total Disponível: {df_candidatos['potencial_mw'].sum()/1000:.2f} GW")

Base carregada com 8,150 latifúndios.
Potencial Total Disponível: 1080.47 GW


In [4]:
# Meta Nacional: 8.600 MW
META_MW = 8600 

## Modelagem do problema de otimização

### Modelo 1 apenas com custo por distância da rede elétrica

In [5]:
# Definição do Problema (Minimização de Custo)
prob = pulp.LpProblem("Otimizacao_Alocacao_Solar_MG", pulp.LpMinimize)

# Parâmetros (PDE 2034)

# Variáveis de Decisão (Xi binário para cada fazenda)
# x[i] = 1 se a fazenda i for escolhida, 0 caso contrário
indices = df_candidatos.index
x = pulp.LpVariable.dicts("fazenda", indices, cat='Binary')

# Função Objetivo: Minimizar o somatório dos Custos de Conexão
prob += pulp.lpSum([df_candidatos.loc[i, 'custo_conexao_rs'] * x[i] for i in indices])

# Restrição: Atingir a Meta de Geração (MW)
prob += pulp.lpSum([df_candidatos.loc[i, 'potencial_mw'] * x[i] for i in indices]) >= META_MW

print(f"Modelo formulado para uma meta de {META_MW} MW.")

Modelo formulado para uma meta de 8600 MW.


### Modelo 2 com custo do terreno, custo fixo de entrada na rede e limite superior de geração por terreno

In [6]:
print("Definindo restrições físicas e custos de mercado...")

# Limite de Injeção: Nenhuma usina pode injetar mais de 300 MW em um único ponto
df_candidatos['potencial_mw_limitado'] = df_candidatos['potencial_mw'].clip(upper=300)

# Custo Fixo de entrada na rede
CUSTO_FIXO_CONEXAO = 5000000 

# Custo Fundiário (Fonte: Reland - Estimativa para MG)
CUSTO_POR_HECTARE = 162178
df_candidatos['custo_terra_rs'] = df_candidatos['area_util_ha'] * CUSTO_POR_HECTARE

# Custo Total do Projeto
df_candidatos['custo_total_projeto_rs'] = df_candidatos['custo_conexao_rs'] + CUSTO_FIXO_CONEXAO + df_candidatos['custo_terra_rs']

# Definição do Problema
prob = pulp.LpProblem("Otimizacao_Alocacao_Solar_MG_Complexo", pulp.LpMinimize)

indices = df_candidatos.index
x = pulp.LpVariable.dicts("fazenda", indices, cat='Binary')

# Função Objetivo: Custo Total do Projeto
prob += pulp.lpSum([df_candidatos.loc[i, 'custo_total_projeto_rs'] * x[i] for i in indices])

# Restrição: Atingir a Meta (usando o potencial limitado)
prob += pulp.lpSum([df_candidatos.loc[i, 'potencial_mw_limitado'] * x[i] for i in indices]) >= META_MW

print(f"Modelo COMPLEXO formulado para uma meta de {META_MW} MW.")

Definindo restrições físicas e custos de mercado...
Modelo COMPLEXO formulado para uma meta de 8600 MW.


## Execução do solver sobre o modelo

In [7]:
# Iniciando o Solver
prob.solve(pulp.PULP_CBC_CMD(msg=1, gapRel=0.01))

print(f"Status da Solução: {pulp.LpStatus[prob.status]}")

# Extrair os resultados
df_candidatos['selecionado'] = [pulp.value(x[i]) for i in indices]
selecionados = df_candidatos[df_candidatos['selecionado'] == 1.0].copy()

if "Complexo" in prob.name:
    potencial_total_mw = selecionados['potencial_mw_limitado'].sum()
    coluna_custo_ordenacao = 'custo_total_projeto_rs'
else:
    potencial_total_mw = selecionados['potencial_mw'].sum()
    coluna_custo_ordenacao = 'custo_conexao_rs'

custo_total_bilhoes = pulp.value(prob.objective) / 1e9

print("-" * 55)
print(f"RESULTADOS DA OTIMIZAÇÃO: {prob.name}")
print("-" * 55)
print(f"Propriedades Selecionadas: {len(selecionados)}")
print(f"Potencial Alocado       : {potencial_total_mw:.2f} MW")
print(f"Custo Total (Objetivo)  : R$ {custo_total_bilhoes:.3f} Bilhões")
print(f"Custo Médio por MW      : R$ {(pulp.value(prob.objective)/potencial_total_mw):,.2f}")
print("-" * 55)

# Visualizar as top propriedades escolhidas (mostrando a coluna correta)
print(f"\nTop 5 Propriedades (Ordenadas por {coluna_custo_ordenacao}):")
colunas_exibicao = ['id_imovel', 'nome_imovel', 'area_util_ha', 'distancia_km', coluna_custo_ordenacao]
print(selecionados[colunas_exibicao].sort_values(by=coluna_custo_ordenacao).head())

Status da Solução: Optimal
-------------------------------------------------------
RESULTADOS DA OTIMIZAÇÃO: Otimizacao_Alocacao_Solar_MG_Complexo
-------------------------------------------------------
Propriedades Selecionadas: 34
Potencial Alocado       : 8613.92 MW
Custo Total (Objetivo)  : R$ 5.227 Bilhões
Custo Médio por MW      : R$ 606,848.39
-------------------------------------------------------

Top 5 Propriedades (Ordenadas por custo_total_projeto_rs):
        id_imovel                                nome_imovel  area_util_ha  \
86   4.380570e+12  Sítio Flor de Minas - Sítio Flor de Minas    756.344070   
136  4.061800e+12              Fazenda Lapa Grande - Parte 2    756.344070   
146  3.511724e+08      FAZENDA PANTANO OU MARIANO - Gleba 04    756.344070   
159  4.101010e+12        FAZENDA POÇÕES - Gleba REMANESCENTE    756.344070   
40   4.060820e+12  FAZENDA BOM SUCESSO - Fazenda Bom Sucesso    850.887079   

     distancia_km  custo_total_projeto_rs  
86       0.183520 

### Solver para os três cenários do PDE

In [9]:
# Definição das Metas (MG assume 20% da meta nacional de 8600 MW)
META_NACIONAL = 8600
COTA_MG = 0.20
META_BASE_MG = META_NACIONAL * COTA_MG

cenarios = {
    "Pessimista (80%)": META_BASE_MG * 0.8,
    "Referência (100%)": META_BASE_MG,
    "Otimista (120%)": META_BASE_MG * 1.2
}

resultados_cenarios = []
indices = df_candidatos.index

for nome_cenario, meta_mw in cenarios.items():
    print(f" A otimizar o cenário: {nome_cenario} (Meta: {meta_mw:.2f} MW)")
    
    # Iniciar o Problema
    prob = pulp.LpProblem(f"Otimizacao_Solar_MG_{nome_cenario[:3]}", pulp.LpMinimize)
    x = pulp.LpVariable.dicts(f"fazenda_{nome_cenario[:3]}", indices, cat='Binary')
    
    # Função Objetivo: Minimizar Custo Total do Projeto (Transmissão + Bay + Terra)
    prob += pulp.lpSum([df_candidatos.loc[i, 'custo_total_projeto_rs'] * x[i] for i in indices])
    
    # Restrição: Atingir a Meta (Potencial limitado a 300 MW por usina)
    prob += pulp.lpSum([df_candidatos.loc[i, 'potencial_mw_limitado'] * x[i] for i in indices]) >= meta_mw
    
    # Configurar o Solver com as heurísticas de paragem (gap de 1%)
    # Usamos msg=0 aqui para manter o ecrã limpo e focar apenas na tabela final
    solver = pulp.PULP_CBC_CMD(msg=0, gapRel=0.01)
    prob.solve(solver)
    
    # Extrair métricas
    selecionados_idx = [i for i in indices if pulp.value(x[i]) == 1.0]
    potencial_alocado = df_candidatos.loc[selecionados_idx, 'potencial_mw_limitado'].sum()
    custo_bilhoes = pulp.value(prob.objective) / 1e9
    
    resultados_cenarios.append({
        "Cenário": nome_cenario,
        "Meta PDE (MW)": meta_mw,
        "Alocado (MW)": potencial_alocado,
        "Fazendas Escolhidas": len(selecionados_idx),
        "Custo Total (R$ Bilhões)": round(custo_bilhoes, 3),
        "Custo Médio (R$ Milhões/MW)": round((custo_bilhoes * 1000) / potencial_alocado, 2)
    })

# Exibição do Balanço Final
print("\n" + "="*85)
print("BALANÇO FINAL DE OTIMIZAÇÃO (CENÁRIOS PDE 2034 - MG)")
print("="*85)
df_resultados = pd.DataFrame(resultados_cenarios)
print(df_resultados.to_string(index=False))
print("="*85)

 A otimizar o cenário: Pessimista (80%) (Meta: 1376.00 MW)
 A otimizar o cenário: Referência (100%) (Meta: 1720.00 MW)
 A otimizar o cenário: Otimista (120%) (Meta: 2064.00 MW)

BALANÇO FINAL DE OTIMIZAÇÃO (CENÁRIOS PDE 2034 - MG)
          Cenário  Meta PDE (MW)  Alocado (MW)  Fazendas Escolhidas  Custo Total (R$ Bilhões)  Custo Médio (R$ Milhões/MW)
 Pessimista (80%)         1376.0   1376.739822                    5                     0.839                         0.61
Referência (100%)         1720.0   1733.288494                    7                     1.049                         0.61
  Otimista (120%)         2064.0   2074.693803                    8                     1.254                         0.60
